# **Cloud-Native GRIB2: High-Performance Access with grib2io and VirtualiZarr**

This notebook demonstrates how to efficiently access GRIB2 data stored on AWS S3 without streaming the entire file. We'll leverage standard `.idx` sidecar files to build a virtual Xarray dataset instantly.

## Notebook Convention

Use the standard grib2io xarray backend interface directly.
Pass grib2io-specific options as keyword arguments to `xr.open_dataset(...)` or `xr.open_mfdataset(...)` (for example `use_icechunk`, `storage_options`, `filters`, `max_workers`, `network_timeout`, and `max_concurrent_requests`).

## **The GRIB2 Challenge**

GRIB2 is a streaming format without a central metadata header. To create a virtual Zarr manifest, a standard scanner must read the file from start to finish to find message boundaries. For a multi-gigabyte file on S3, this normally means streaming all that data just to find the offsets.

### **The Solution: Standard sidecar Indices**
Public datasets (like NOAA GFS) often provide text-based `.idx` files alongside the GRIB files. `grib2io` can now parse these `.idx` files to extract byte offsets directly. This allows it to jump to each message and read only the small metadata headers (Sections 0-5), completely bypassing the large data payloads (Section 7) during the scanning phase.

In [ ]:
import xarray as xr
import grib2io
import fsspec
from pathlib import Path
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import KerchunkJSONParser
from obstore.store import LocalStore, S3Store
from obspec_utils.registry import ObjectStoreRegistry
import s3fs
import json
import os

## **1. Locating GFS Data on S3**

We'll use a public GFS dataset from the NOAA Open Data Program on AWS. Note the presence of the `.idx` file.

In [ ]:
# Example S3 path for a GFS file
s3_bucket = "noaa-gfs-bdp-pds"
s3_path = "gfs.20240501/00/atmos/gfs.t00z.pgrb2.0p25.f000"
s3_url = f"s3://{s3_bucket}/{s3_path}"
# Standard sidecar index: s3://noaa-gfs-bdp-pds/gfs.20240501/00/atmos/gfs.t00z.pgrb2.0p25.f000.idx

# Verify the key exists in the bucket before trying to build references.
fs = s3fs.S3FileSystem(anon=True)
if not fs.exists(f"{s3_bucket}/{s3_path}"):
    raise FileNotFoundError(f"S3 object not found: {s3_url}")

## **2. Building the Virtual Manifest via Index**

By leveraging the `.idx` file, `grib2io` maps out the chunks instantly. The `ReferenceGenerator` uses this to create a Kerchunk manifest.

In [ ]:
from grib2io.kerchunk import ReferenceGenerator

# Stream directly from S3 (no local download).
gen = ReferenceGenerator(
    s3_url,
    storage_options={
        "anon": True,
        "default_fill_cache": False,
        "default_cache_type": "none",
        "default_block_size": 131072,
    },
)

# This step is now extremely fast as it avoids streaming data payloads.
manifest = gen.generate()

manifest_path = "gfs_s3_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f)

## **3. Ingesting into VirtualiZarr**

Load the S3-backed manifest into VirtualiZarr for metadata manipulation.

In [ ]:
manifest_file = Path(manifest_path).resolve()
manifest_url = manifest_file.as_uri()

registry = ObjectStoreRegistry(
    {
        manifest_url: LocalStore(prefix=str(manifest_file.parent)),
        "s3://noaa-gfs-bdp-pds": S3Store(bucket="noaa-gfs-bdp-pds", skip_signature=True),
    }
)

vds = open_virtual_dataset(
    url=manifest_url,
    registry=registry,
    parser=KerchunkJSONParser(),
    loadable_variables=[],
)
display(vds)

## **4. Targeted Data Access**

Use the standard grib2io/xarray interface for dataset opening:

| Interface | Input | Use when… |
|---|---|---|
| `xr.open_dataset(url, engine="grib2io", use_icechunk=True, ...)` | GRIB2 file path / URI | You want a single call from file → Dataset |
| `xr.open_mfdataset(urls, engine="grib2io", use_icechunk=True, ...)` | list of GRIB2 paths / URIs | You want a combined multi-file Dataset |
| `xr.open_dataset(manifest_path, engine="grib2io")` | pre-built manifest file | You already have a saved manifest |

Notebook convention: pass grib2io-specific options directly as keyword arguments
to `xr.open_dataset(...)` / `xr.open_mfdataset(...)`.

All three paths route through the same backend UI and handle local/remote storage transparently.


In [ ]:
# ── Option A: one call from file path -> Dataset via grib2io backend ─────────
ds = xr.open_dataset(
    s3_url,
    engine="grib2io",
    use_icechunk=True,
    storage_options={"anon": True},
)

# ── Option B: if you already built (or loaded) a manifest file ───────────────
# ds = xr.open_dataset(manifest_path, engine="grib2io")

display(ds)

In [ ]:
vds

In [ ]:
if "TMP" in ds.data_vars:
    data = ds.TMP.isel(valid_time=0, isobaric_surface=0, y=slice(100, 200), x=slice(100, 200)).compute()
    print(f"shape: {data.shape}, min: {float(data.min()):.2f} K, max: {float(data.max()):.2f} K")
    display(data)